# 1. Setup Environment

In [ ]:
!git clone https://github.com/clovaai/deep-text-recognition-benchmark
%cd deep-text-recognition-benchmark
!pip install lmdb pillow torchvision nltk natsort torch

# 2. Upload and Extract Dataset

In [ ]:
# ============================================================
# Cell 3 - Upload & unpack the synthetic+real word-level dataset
# (drift-resistant set: 8k synthetic + 6.7k real = 45% real; train=14672 / val=123)
# ============================================================
from google.colab import files
import os

ZIP = "easyocr_dataset_synth_ready.zip"

print(f"Please upload {ZIP} ...")
uploaded = files.upload()

if ZIP not in uploaded:
    raise SystemExit(f"ERROR: expected {ZIP}, got {list(uploaded)}. Re-run and pick {ZIP}.")

print("Upload complete! Extracting...")
!unzip -q -o easyocr_dataset_synth_ready.zip

# The zip holds train_lmdb/ and val_lmdb/ at its root -> move under dataset/
!mkdir -p dataset
!rm -rf dataset/train_lmdb dataset/val_lmdb
!mv train_lmdb dataset/ 2>/dev/null || true
!mv val_lmdb   dataset/ 2>/dev/null || true

# Sanity-check both LMDBs landed and report sample counts
import lmdb
for split in ["train_lmdb", "val_lmdb"]:
    path = f"dataset/{split}"
    if not os.path.exists(os.path.join(path, "data.mdb")):
        raise SystemExit(f"ERROR: {path}/data.mdb missing after extract.")
    env = lmdb.open(path, readonly=True, lock=False)
    with env.begin() as txn:
        n = txn.get(b"num-samples").decode()
    print(f"  {split}: {n} samples")
    env.close()

print("Ready - dataset/train_lmdb + dataset/val_lmdb in place. Proceed to the train cell.")


# 3. Download original weights, Patch bug, and Fine-Tune

## Upload base model (deployed apex.pth) to continue from

In [ ]:
# ============================================================
# Upload the base model to CONTINUE from = your deployed apex.pth (89.4%)
# (on your machine: models/easyocr_custom/apex.pth)
# ============================================================
from google.colab import files
import os
print("Upload apex.pth (your current deployed 89.4% model)...")
up = files.upload()
assert "apex.pth" in up, f"Expected apex.pth, got {list(up)}"
print("apex.pth ready:", os.path.getsize("apex.pth"), "bytes")


In [ ]:
import subprocess
import sys
import os
import urllib.request
import zipfile

# 1. Download the actual pre-trained english_g2 model
if False:  # base model is the uploaded apex.pth, not stock english_g2
    print("Downloading pretrained english_g2 model...")
    urllib.request.urlretrieve("https://github.com/JaidedAI/EasyOCR/releases/download/v1.3/english_g2.zip", "english_g2.zip")
    with zipfile.ZipFile("english_g2.zip", 'r') as zip_ref:
        zip_ref.extractall(".")
    print("Extracted english_g2.pth!")

# 2. Patch train.py so it doesn't overwrite our custom characters and works on CPU/GPU
print("Patching train.py...")
with open("train.py", "r", encoding="utf-8") as f:
    code = f.read()
# Disable the hardcoded overwrite
code = code.replace("opt.character = string.printable[:-6]", "pass # opt.character OVERWRITE DISABLED")
# Fix CUDA deserialization issue if running on CPU
code = code.replace("torch.load(opt.saved_model)", "torch.load(opt.saved_model, map_location=device)")
with open("train.py", "w", encoding="utf-8") as f:
    f.write(code)

# 3. Patch dataset.py to fix PyTorch _accumulate import error!
print("Patching dataset.py...")
with open("dataset.py", "r", encoding="utf-8") as f:
    code = f.read()
code = code.replace("from torch._utils import _accumulate", "from itertools import accumulate as _accumulate")
with open("dataset.py", "w", encoding="utf-8") as f:
    f.write(code)

# 3. The EXACT character string used by JaidedAI (including the degree symbol °)
chars = "0123456789!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~ °ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"

cmd = [
    "python", "train.py",
    "--train_data", "dataset/train_lmdb",
    "--valid_data", "dataset/val_lmdb",
    "--select_data", "/",
    "--batch_ratio", "1",
    "--Transformation", "None",
    "--FeatureExtraction", "VGG",
    "--SequenceModeling", "BiLSTM",
    "--Prediction", "CTC",
    "--saved_model", "apex.pth", # Use the fully pretrained model
    "--FT",                            # Fine-tune mode (keeps prediction weights!)
    "--output_channel", "256",         # Original model dimensions
    "--hidden_size", "256",            # Original model dimensions
    "--batch_size", "128",
    "--data_filtering_off",
    "--workers", "4",
    "--num_iter", "2000",              # Only need 5000 steps since we're fine-tuning
    "--lr", "0.1",                     # Default (1.0) is tuned for from-scratch training on
                                        # millions of samples; kept low so gradients aren't
                                        # violent relative to this dataset's size.
    "--imgW", "512",                   # Wide enough that CTC's ~127 timesteps comfortably
                                        # exceed the longest word-level label (max 40 chars).
    "--PAD",                           # Resize proportionally to imgH then right-pad to imgW,
                                        # instead of squashing to a fixed box. Matches how
                                        # EasyOCR resizes crops at inference time.
    "--valInterval", "100",            # Finer-grained checkpoints so the actual best iteration
                                        # can be pinpointed instead of only sampling every 500.
    "--exp_name", "apex_wordlevel_continue",
    "--sensitive",
    "--batch_max_length", "256",
    "--character", chars
]

# NOTE: this dataset (easyocr_lmdb_wordlevel_ready, uploaded as easyocr_lmdb_dataset_ready.zip)
# is built differently from the first two training attempts:
#   1. Word-level crops, not whole killfeed lines. ocr_with_easyocr() at inference recognizes
#      each CRAFT-detected word box separately -- it never sees a whole line, and never emits
#      "<GUN_ICON>" itself (that marker is inserted afterward by a Python-side pixel-gap
#      heuristic). Training on whole lines labeled with the literal "<GUN_ICON>" text taught
#      the model a task it's never actually asked to perform at inference. These crops are
#      individual CRAFT-detected word boxes with the marker text removed from labels entirely.
#   2. Correct color polarity. The previous LMDB-build script copied raw crop files directly
#      (light background, dark text). Inference always runs preprocess_for_easyocr() first,
#      which inverts colors (dark background, light text) before OCR. These crops are taken
#      from the *preprocessed* image, so training now sees the same polarity as inference.
# Both were confirmed via direct pixel inspection, not assumption -- see prepare_wordlevel_dataset.py.

print("Starting fine-tuning! Output will stream below...")
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in process.stdout:
    sys.stdout.write(line)
    sys.stdout.flush()
process.wait()


# 4. Download Trained Weights

In [ ]:
from google.colab import files
# best_norm_ED is the checkpoint that won last time (label-noise makes best_accuracy pick too early)
files.download('saved_models/apex_wordlevel_continue/best_norm_ED.pth')
files.download('saved_models/apex_wordlevel_continue/best_accuracy.pth')
